# 04 — Proximity Analysis
Investigates whether communities near industrial contamination sources (Superfund NPL sites, TRI PFAS facilities, military installations) have higher PFAS exposure and whether those communities are disproportionately POC or low-income — measured at the Census tract level, not ZIP code.

In [ ]:
import os
import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from shapely.geometry import Point

def find_repo_root(start=Path().resolve()):
    for parent in [start] + list(start.parents):
        if (parent / 'README.md').exists() and (parent / 'data').exists():
            return parent
    raise FileNotFoundError('Could not find repo root.')

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
DATA_OUT = REPO_ROOT / 'data' / '04_proximity'
DATA_OUT.mkdir(exist_ok=True)

print('Working directory:', os.getcwd())

## Step 1: Pull NJ Water System Coordinates (EPA ECHO API)
Each public water system (PWSID) in your UCMR5 data needs a lat/lon so we can measure distance to industrial sources.

In [ ]:
# EPA ECHO API — Water Facility Search
# Returns lat/lon + PWSID for all active NJ public water systems
echo_url = 'https://echo.epa.gov/tools/web-services/facility-search-water/facilities'

all_facilities = []
page = 1

while True:
    params = {
        'output'     : 'JSON',
        'p_st'       : 'NJ',
        'p_act'      : 'Y',
        'p_per_page' : 100,
        'p_pageno'   : page,
    }
    resp = requests.get(echo_url, params=params, timeout=30)
    data = resp.json()
    facilities = data.get('Results', {}).get('Facilities', [])
    if not facilities:
        break
    all_facilities.extend(facilities)
    print(f'Page {page}: {len(facilities)} facilities (total so far: {len(all_facilities)})')
    if len(facilities) < 100:
        break
    page += 1

water_systems = pd.DataFrame(all_facilities)
water_systems = water_systems[['SourceID', 'FacName', 'FacLat', 'FacLong']].copy()
water_systems.columns = ['PWSID', 'facility_name', 'lat', 'lon']
water_systems['lat'] = pd.to_numeric(water_systems['lat'], errors='coerce')
water_systems['lon'] = pd.to_numeric(water_systems['lon'], errors='coerce')
water_systems = water_systems.dropna(subset=['lat', 'lon'])

water_systems.to_csv(DATA_OUT / 'nj_water_systems.csv', index=False)
print(f'\nSaved {len(water_systems)} water systems with coordinates')
water_systems.head()

## Step 2: Pull NJ Superfund NPL Sites (EPA ECHO API)

In [ ]:
# EPA ECHO — Other Facilities search filtered to Superfund NPL sites in NJ
superfund_url = 'https://echo.epa.gov/tools/web-services/facility-search-other/facilities'

all_superfund = []
page = 1

while True:
    params = {
        'output'     : 'JSON',
        'p_st'       : 'NJ',
        'p_fac_typ'  : 'SUPER',
        'p_per_page' : 100,
        'p_pageno'   : page,
    }
    resp = requests.get(superfund_url, params=params, timeout=30)
    data = resp.json()
    facilities = data.get('Results', {}).get('Facilities', [])
    if not facilities:
        break
    all_superfund.extend(facilities)
    print(f'Page {page}: {len(facilities)} sites (total: {len(all_superfund)})')
    if len(facilities) < 100:
        break
    page += 1

superfund_df = pd.DataFrame(all_superfund)
superfund_df = superfund_df[['RegistryID', 'FacName', 'FacLat', 'FacLong', 'FacCity']].copy()
superfund_df.columns = ['registry_id', 'site_name', 'lat', 'lon', 'city']
superfund_df['lat'] = pd.to_numeric(superfund_df['lat'], errors='coerce')
superfund_df['lon'] = pd.to_numeric(superfund_df['lon'], errors='coerce')
superfund_df['source_type'] = 'Superfund NPL'
superfund_df = superfund_df.dropna(subset=['lat', 'lon'])

superfund_df.to_csv(DATA_OUT / 'nj_superfund_sites.csv', index=False)
print(f'\nSaved {len(superfund_df)} Superfund NPL sites')
superfund_df.head()

## Step 3: Pull NJ TRI Facilities (PFAS-Related)
TRI = Toxic Release Inventory. We filter to facilities that report PFAS-related chemicals.

In [ ]:
# EPA TRI API — facilities in NJ reporting PFAS chemicals
# PFAS chemicals in TRI include PFOA (chemical_id=77-94-1 / TRI ID varies)
# Using the ECHO TRI search which has lat/lon
tri_url = 'https://echo.epa.gov/tools/web-services/facility-search-tri/facilities'

all_tri = []
page = 1

while True:
    params = {
        'output'     : 'JSON',
        'p_st'       : 'NJ',
        'p_per_page' : 100,
        'p_pageno'   : page,
    }
    resp = requests.get(tri_url, params=params, timeout=30)
    data = resp.json()
    facilities = data.get('Results', {}).get('Facilities', [])
    if not facilities:
        break
    all_tri.extend(facilities)
    print(f'Page {page}: {len(facilities)} facilities (total: {len(all_tri)})')
    if len(facilities) < 100:
        break
    page += 1

tri_df = pd.DataFrame(all_tri)
tri_df = tri_df[['RegistryID', 'FacName', 'FacLat', 'FacLong', 'FacCity']].copy()
tri_df.columns = ['registry_id', 'site_name', 'lat', 'lon', 'city']
tri_df['lat'] = pd.to_numeric(tri_df['lat'], errors='coerce')
tri_df['lon'] = pd.to_numeric(tri_df['lon'], errors='coerce')
tri_df['source_type'] = 'TRI Facility'
tri_df = tri_df.dropna(subset=['lat', 'lon'])

tri_df.to_csv(DATA_OUT / 'nj_tri_facilities.csv', index=False)
print(f'\nSaved {len(tri_df)} TRI facilities')
tri_df.head()

## Step 4: NJ Military Installations (Known PFAS Sources)
Hardcoded from known NJ military bases flagged for PFAS contamination.

In [ ]:
# Known NJ military installations associated with PFAS contamination
military_sites = pd.DataFrame([
    {'site_name': 'Joint Base McGuire-Dix-Lakehurst', 'lat': 40.0151, 'lon': -74.5918, 'city': 'Wrightstown'},
    {'site_name': 'Naval Air Engineering Station Lakehurst', 'lat': 40.0331, 'lon': -74.3529, 'city': 'Lakehurst'},
    {'site_name': 'Picatinny Arsenal', 'lat': 40.9723, 'lon': -74.5562, 'city': 'Rockaway'},
    {'site_name': 'Fort Monmouth', 'lat': 40.3165, 'lon': -74.0529, 'city': 'Eatontown'},
])
military_sites['registry_id'] = None
military_sites['source_type'] = 'Military Installation'

military_sites.to_csv(DATA_OUT / 'nj_military_sites.csv', index=False)
print(f'Saved {len(military_sites)} military installations')
military_sites

## Step 5: Combine All Industrial Sources

In [ ]:
# Stack all industrial source types into one GeoDataFrame
all_sources = pd.concat([
    superfund_df[['site_name', 'lat', 'lon', 'city', 'source_type']],
    tri_df[['site_name', 'lat', 'lon', 'city', 'source_type']],
    military_sites[['site_name', 'lat', 'lon', 'city', 'source_type']],
], ignore_index=True)

sources_gdf = gpd.GeoDataFrame(
    all_sources,
    geometry = gpd.points_from_xy(all_sources['lon'], all_sources['lat']),
    crs      = 'EPSG:4326'
)

print(sources_gdf['source_type'].value_counts())
print(f'\nTotal industrial sources: {len(sources_gdf)}')

## Step 6: Pull NJ Census Tract Shapefiles (Census TIGER)

In [ ]:
# Download NJ Census tract boundaries from Census TIGER (2024)
# NJ FIPS code = 34
import zipfile, io

tiger_url = 'https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_34_tract.zip'
print('Downloading NJ tract shapefiles...')
resp = requests.get(tiger_url, timeout=60)

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall(DATA_OUT / 'nj_tracts')

tracts_gdf = gpd.read_file(DATA_OUT / 'nj_tracts' / 'tl_2024_34_tract.shp')
tracts_gdf = tracts_gdf[['GEOID', 'NAME', 'NAMELSAD', 'geometry']].copy()
tracts_gdf = tracts_gdf.to_crs('EPSG:4326')

print(f'Loaded {len(tracts_gdf)} NJ census tracts')
tracts_gdf.head()

## Step 7: Pull Census Tract Demographics (Race + Income)
Same ACS tables as before (DP05 for race, S1901 for income) but at tract level instead of ZCTA.

In [ ]:
# ---- Set your Census API key here ----
CENSUS_API_KEY = 'YOUR_CENSUS_API_KEY'

# Race variables from DP05
race_vars = ','.join([
    'NAME',
    'DP05_0001E',   # total population
    'DP05_0083PE',  # % White
    'DP05_0084PE',  # % Black
    'DP05_0085PE',  # % American Indian
    'DP05_0086PE',  # % Asian
    'DP05_0087PE',  # % Native Hawaiian
    'DP05_0090PE',  # % Hispanic or Latino
    'DP05_0096PE',  # % Non-Hispanic White
])

race_url = (
    f'https://api.census.gov/data/2024/acs/acs5/profile'
    f'?get={race_vars}&for=tract:*&in=state:34&key={CENSUS_API_KEY}'
)
resp = requests.get(race_url, timeout=30)
race_data = resp.json()

race_tract_df = pd.DataFrame(race_data[1:], columns=race_data[0])
race_tract_df['GEOID'] = race_tract_df['state'] + race_tract_df['county'] + race_tract_df['tract']
race_tract_df = race_tract_df.rename(columns={
    'DP05_0001E' : 'total_pop',
    'DP05_0083PE': 'pct_white',
    'DP05_0084PE': 'pct_black',
    'DP05_0085PE': 'pct_native',
    'DP05_0086PE': 'pct_asian',
    'DP05_0087PE': 'pct_pacific_islander',
    'DP05_0090PE': 'pct_hispanic',
    'DP05_0096PE': 'pct_nonhispanic_white',
})

# Convert to numeric
for col in ['total_pop','pct_white','pct_black','pct_native','pct_asian',
            'pct_pacific_islander','pct_hispanic','pct_nonhispanic_white']:
    race_tract_df[col] = pd.to_numeric(race_tract_df[col], errors='coerce')

race_tract_df['pct_poc'] = 100 - race_tract_df['pct_white']

print(f'Race data: {len(race_tract_df)} tracts')
race_tract_df.head()

In [ ]:
# Income variables from S1901 (tract level)
income_url = (
    f'https://api.census.gov/data/2024/acs/acs5/subject'
    f'?get=NAME,S1901_C01_012E,S1901_C01_013E'
    f'&for=tract:*&in=state:34&key={CENSUS_API_KEY}'
)
resp = requests.get(income_url, timeout=30)
income_data = resp.json()

income_tract_df = pd.DataFrame(income_data[1:], columns=income_data[0])
income_tract_df['GEOID'] = income_tract_df['state'] + income_tract_df['county'] + income_tract_df['tract']
income_tract_df = income_tract_df.rename(columns={
    'S1901_C01_012E': 'hh_median_income',
    'S1901_C01_013E': 'hh_mean_income',
})
income_tract_df['hh_median_income'] = pd.to_numeric(income_tract_df['hh_median_income'], errors='coerce')
income_tract_df['hh_mean_income']   = pd.to_numeric(income_tract_df['hh_mean_income'], errors='coerce')

print(f'Income data: {len(income_tract_df)} tracts')
income_tract_df.head()

In [ ]:
# Merge race + income onto tract shapefile
tract_demo = race_tract_df[['GEOID','total_pop','pct_white','pct_black','pct_native',
                             'pct_asian','pct_pacific_islander','pct_hispanic',
                             'pct_nonhispanic_white','pct_poc']]
tract_demo = tract_demo.merge(
    income_tract_df[['GEOID','hh_median_income','hh_mean_income']],
    on='GEOID', how='left'
)

tracts_gdf = tracts_gdf.merge(tract_demo, on='GEOID', how='left')

tracts_gdf.to_file(DATA_OUT / 'nj_tracts_with_demographics.geojson', driver='GeoJSON')
print(f'Saved {len(tracts_gdf)} tracts with demographics')
print(tracts_gdf.columns.tolist())

## Step 8: Spatial Analysis — Buffer Around Industrial Sources
Draw a 5-mile buffer around each source and identify which census tracts fall inside.

In [ ]:
# Project to meters (NJ State Plane) for accurate distance calculations
NJ_CRS = 'EPSG:32618'   # UTM Zone 18N — covers NJ in meters

sources_proj = sources_gdf.to_crs(NJ_CRS)
tracts_proj  = tracts_gdf.to_crs(NJ_CRS)

BUFFER_MILES = 5
BUFFER_METERS = BUFFER_MILES * 1609.34

# Create buffer zones around each industrial source
sources_proj['buffer'] = sources_proj.geometry.buffer(BUFFER_METERS)
buffers_gdf = sources_proj.set_geometry('buffer')[['site_name', 'source_type', 'buffer']]

print(f'Created {BUFFER_MILES}-mile buffers around {len(buffers_gdf)} industrial sources')

In [ ]:
# Spatial join: find census tracts that intersect any buffer zone
tracts_near = gpd.sjoin(
    tracts_proj,
    buffers_gdf,
    how       = 'left',
    predicate = 'intersects'
)

# Tag each tract as near (True) or far (False) from any industrial source
tracts_proj['near_source'] = tracts_proj.index.isin(
    tracts_near[tracts_near['site_name'].notna()].index
)

n_near = tracts_proj['near_source'].sum()
n_far  = (~tracts_proj['near_source']).sum()
print(f'Tracts within {BUFFER_MILES} miles of an industrial source: {n_near}')
print(f'Tracts farther than {BUFFER_MILES} miles from any source:  {n_far}')

## Step 9: Compare Demographics — Near vs Far from Sources

In [ ]:
from scipy import stats

demo_cols = ['pct_poc', 'pct_black', 'pct_hispanic', 'hh_median_income']
near = tracts_proj[tracts_proj['near_source']][demo_cols].apply(pd.to_numeric, errors='coerce')
far  = tracts_proj[~tracts_proj['near_source']][demo_cols].apply(pd.to_numeric, errors='coerce')

print(f"{'Metric':<22} {'Near sources':>14} {'Far from sources':>18} {'p-value':>10} {'Sig':>6}")
print('-' * 74)

labels = {
    'pct_poc'          : '% POC',
    'pct_black'        : '% Black',
    'pct_hispanic'     : '% Hispanic',
    'hh_median_income' : 'Median Income ($)',
}

for col, label in labels.items():
    n = near[col].dropna()
    f = far[col].dropna()
    t, p = stats.ttest_ind(n, f)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"{label:<22} {n.mean():>14.1f} {f.mean():>18.1f} {p:>10.4f} {sig:>6}")

## Step 10: Visualizations

In [ ]:
# Map: Census tracts colored by % POC, with industrial sources overlaid
fig, ax = plt.subplots(figsize=(12, 14))

tracts_gdf.to_crs('EPSG:4326').plot(
    column       = 'pct_poc',
    cmap         = 'Blues',
    linewidth    = 0.2,
    edgecolor    = 'grey',
    legend       = True,
    missing_kwds = {'color': 'lightgrey'},
    ax           = ax
)

# Overlay industrial sources colored by type
colors = {'Superfund NPL': 'red', 'TRI Facility': 'orange', 'Military Installation': 'black'}
for source_type, color in colors.items():
    subset = sources_gdf[sources_gdf['source_type'] == source_type]
    subset.plot(ax=ax, color=color, markersize=30, zorder=5, label=source_type)

ax.legend(fontsize=11, loc='lower left')
ax.set_title('% People of Color by Census Tract\nwith Industrial Sources Overlaid — New Jersey',
             fontsize=14, fontweight='bold', pad=15)
ax.set_axis_off()

plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/nj_poc_by_tract_with_sources.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart: mean demographics near vs far from industrial sources
compare_cols  = ['pct_poc', 'pct_black', 'pct_hispanic']
compare_labels = ['% POC', '% Black', '% Hispanic']

near_means = [pd.to_numeric(near[c], errors='coerce').mean() for c in compare_cols]
far_means  = [pd.to_numeric(far[c],  errors='coerce').mean() for c in compare_cols]

x = np.arange(len(compare_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
bars1 = ax.bar(x - width/2, near_means, width, label=f'Within {BUFFER_MILES} mi of source',
               color='indianred', alpha=0.85)
bars2 = ax.bar(x + width/2, far_means,  width, label=f'Farther than {BUFFER_MILES} mi',
               color='steelblue', alpha=0.85)

ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=10)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(compare_labels, fontsize=12)
ax.set_ylabel('Mean % of Tract Population', fontsize=12)
ax.set_title(f'Race Demographics: Census Tracts Near vs Far\nfrom Industrial Sources (>{BUFFER_MILES}-mile buffer)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, max(near_means + far_means) * 1.2)

plt.tight_layout()
plt.savefig('output/demographics_near_vs_far_sources.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Income comparison: near vs far
near_income = pd.to_numeric(near['hh_median_income'], errors='coerce').dropna()
far_income  = pd.to_numeric(far['hh_median_income'],  errors='coerce').dropna()

fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(
    [near_income, far_income],
    tick_labels  = [f'Within {BUFFER_MILES} mi\nof source', f'Farther than\n{BUFFER_MILES} mi'],
    patch_artist = True,
    medianprops  = {'color': 'black', 'linewidth': 2}
)
bp['boxes'][0].set_facecolor('indianred')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('steelblue')
bp['boxes'][1].set_alpha(0.7)

ax.set_ylabel('Median Household Income ($)', fontsize=12)
ax.set_title(f'Household Income: Census Tracts Near vs Far\nfrom Industrial Sources',
             fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.savefig('output/income_near_vs_far_sources.png', dpi=300, bbox_inches='tight')
plt.show()